# LSTM — UNIFIED fake detector (CG + human deception)
**Model:** Bidirectional LSTM (128) + GloVe 300d + GlobalMaxPooling1D (same recipe as the broadened CG/OR LSTM).

**Task:** flag a review fake if it is **either computer-generated (CG) OR human-deceptive**. Trained on `combined_fake_reviews.csv` (~130k rows) — large enough for a from-scratch BiLSTM (the human-only set at 2.3k was not). Human-origin TRAIN rows are **oversampled 15×** (val/test untouched). Label: 1 = fake (CG or human), 0 = genuine.

### Instructions
1. **Runtime → Change runtime type → T4/A100 GPU.**
2. Upload `data/combined_fake_reviews.csv` to your Drive root.
3. Run all. GloVe (~822 MB) downloads automatically.
4. Download `combined_fake_lstm_onnx.zip`, extract to `data/combined_fake_lstm_onnx/`.

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
!pip install -q tf2onnx onnxruntime

In [ ]:
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
    print('GloVe 300d downloaded.')
else:
    print('GloVe 300d already present.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SPLIT_CSV = '/content/drive/MyDrive/combined_fake_reviews.csv'

In [ ]:
import random, json, numpy as np, pandas as pd
from sklearn.metrics import classification_report, f1_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
SEED = 42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
MAX_LEN, VOCAB_SIZE, EMBED_DIM, LSTM_UNITS, BATCH_SIZE, EPOCHS, LR = 250, 50_000, 300, 128, 256, 30, 5e-4
GLOVE_PATH, HUMAN_OVERSAMPLE = 'glove.6B.300d.txt', 15

In [ ]:
df = pd.read_csv(SPLIT_CSV); df['text'] = df['text'].astype(str)
tr, va, te = (df[df.split==s] for s in ('train','val','test'))
# oversample human-origin TRAIN rows so the LSTM learns human deception, not just CG
hum = tr[tr.origin=='human']
tr = pd.concat([tr] + [hum]*(HUMAN_OVERSAMPLE-1), ignore_index=True).sample(frac=1, random_state=SEED)
X_train, y_train = tr['text'].tolist(), tr['label'].tolist()
X_val,   y_val   = va['text'].tolist(), va['label'].tolist()
X_test,  y_test  = te['text'].tolist(), te['label'].tolist()
ft_test = te['fake_type'].values
print(f'Train {len(X_train):,} ({HUMAN_OVERSAMPLE}x human oversample) | Val {len(X_val):,} | Test {len(X_test):,}')
print('test fake_type:', dict(te['fake_type'].value_counts()))

In [ ]:
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
def encode(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN, padding='post', truncating='post')
X_train_seq, X_val_seq, X_test_seq = encode(X_train), encode(X_val), encode(X_test)
y_train_arr, y_val_arr, y_test_arr = np.array(y_train), np.array(y_val), np.array(y_test)
print('shapes:', X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)

In [ ]:
glove = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        p = line.split(); glove[p[0]] = np.array(p[1:], dtype=np.float32)
print(f'Loaded {len(glove):,} GloVe vectors')
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
hits = 0
for w, i in tokenizer.word_index.items():
    if i < VOCAB_SIZE and (v := glove.get(w)) is not None:
        embedding_matrix[i] = v; hits += 1
print(f'Coverage: {hits:,} / {min(len(tokenizer.word_index), VOCAB_SIZE):,}')

In [ ]:
inputs = Input(shape=(MAX_LEN,), name='input_layer')
x = Embedding(VOCAB_SIZE, EMBED_DIM, weights=[embedding_matrix], trainable=True)(inputs)
x = SpatialDropout1D(0.3)(x)
x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True, kernel_regularizer=l2(1e-4)))(x)
x = GlobalMaxPooling1D()(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs, outputs)
model.compile(optimizer=Adam(LR), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
cb = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
      ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)]
history = model.fit(X_train_seq, y_train_arr, validation_data=(X_val_seq, y_val_arr),
                    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=cb)

In [ ]:
val_probs = model.predict(X_val_seq, batch_size=BATCH_SIZE).flatten()
best_thr, best_f1 = 0.5, 0.0
for t in np.arange(0.10, 0.90, 0.01):
    f = f1_score(y_val_arr, (val_probs >= t).astype(int), average='macro', zero_division=0)
    if f > best_f1: best_f1, best_thr = f, round(float(t), 2)
print(f'Best threshold (macro F1): {best_thr:.2f}  (val {best_f1:.1%})')
y_pred = (model.predict(X_test_seq, batch_size=BATCH_SIZE).flatten() >= best_thr).astype(int)
print(classification_report(y_test_arr, y_pred, target_names=['Genuine','Fake']))
# the key check: does ONE model catch BOTH fake types?
cg = (y_pred[ft_test=='cg_fake']==1).mean(); hu = (y_pred[ft_test=='human_fake']==1).mean(); ge = (y_pred[ft_test=='genuine']==0).mean()
print(f'PER-FAKE-TYPE  ->  CG-fake recall {cg:.1%} | human-fake recall {hu:.1%} | genuine specificity {ge:.1%}')

In [ ]:
with open('combined_fake_lstm_tokenizer.json', 'w', encoding='utf-8') as f: f.write(tokenizer.to_json())
with open('combined_fake_lstm_threshold.json', 'w') as f: json.dump({'threshold': best_thr}, f, indent=2)
model.save('combined_fake_lstm.keras')
print('saved keras + tokenizer + threshold')

In [ ]:
import subprocess, os
open('cpu_export.py','w').write("import os\nos.environ['CUDA_VISIBLE_DEVICES']=''\nimport tensorflow as tf\ntf.keras.models.load_model('combined_fake_lstm.keras').export('combined_fake_lstm_savedmodel')\nprint('CPU SavedModel done')\n")
print(subprocess.run(['python','cpu_export.py'], capture_output=True, text=True, env={**os.environ,'CUDA_VISIBLE_DEVICES':''}).stdout)
r = subprocess.run(['python','-m','tf2onnx.convert','--saved-model','combined_fake_lstm_savedmodel','--output','combined_fake_lstm.onnx','--opset','13'], capture_output=True, text=True)
print(r.stdout[-500:]); print('ERR' , r.stderr[-300:] if r.returncode else 'ok')

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession('combined_fake_lstm.onnx', providers=['CPUExecutionProvider'])
print('inputs', [i.name for i in sess.get_inputs()], '| outputs', [o.name for o in sess.get_outputs()])
print('sample preds:', sess.run(None, {sess.get_inputs()[0].name: X_test_seq[:4].astype(np.float32)})[0].flatten())

In [ ]:
import zipfile
from google.colab import files
with zipfile.ZipFile('combined_fake_lstm_onnx.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write('combined_fake_lstm.onnx'); z.write('combined_fake_lstm_tokenizer.json'); z.write('combined_fake_lstm_threshold.json')
files.download('combined_fake_lstm_onnx.zip')